In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path
import sys

project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root))

In [3]:
READ_CSV="../../data/interim/data_travel_insurance_interim.csv"
RANDOM_STATE=42
SAVE_RESULT = "results/baseline.csv"

In [4]:
from src.utils import benchmark_models

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, TargetEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

from feature_engine.outliers import Winsorizer

In [5]:
df = pd.read_csv(READ_CSV)
df.head()

,Agency,Agency Type,Distribution Channel,Product Name,Duration,Destination,Net Sales,Commission,Age,Claim,Is Refund,Suspected Fraud,Commission Rate
0,C2B,Airlines,Online,Annual Silver Plan,365,SINGAPORE,216.0,54.0,57,0,No,No,0.25
1,EPX,Travel Agency,Online,Cancellation Plan,4,MALAYSIA,10.0,0.0,33,0,No,No,0.00
2,JZI,Airlines,Online,Basic Plan,19,INDIA,22.0,7.7,26,0,No,No,0.35
3,EPX,Travel Agency,Online,2 way Comprehensive Plan,20,UNITED STATES,112.0,0.0,59,0,No,No,0.00
4,C2B,Airlines,Online,Bronze Plan,8,SINGAPORE,16.0,4.0,28,0,No,No,0.25


In [6]:
x = df.drop(columns=["Claim"])
y = df["Claim"]


In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

In [8]:
NUMERIC_COLS = [features for features in x_train.columns if x_train[features].dtypes != "O"]
CATEGORICAL_COLS = [features for features in x_train.columns if x_train[features].dtypes == "O"]


In [9]:
numeric_pipeline = Pipeline([
    ("winsorizer_iqr", Winsorizer(capping_method="iqr", fold=1.5)),
    ("RobustScaler", RobustScaler()),
])

categorical_ohe_pipeline = Pipeline([
     ("OneHotEncoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, drop="first"))
])

In [10]:


preprocessor = ColumnTransformer(
    [
        ("numeric_pipeline", numeric_pipeline, NUMERIC_COLS),
        ("categorical_ohe_pipeline", categorical_ohe_pipeline, CATEGORICAL_COLS),
    ],
    remainder="drop"
)

In [11]:
pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression())
    ]
)

In [12]:
logreg_base = LogisticRegression(random_state=RANDOM_STATE)
logreg_lasso = LogisticRegression(penalty="l1", solver="liblinear", random_state=RANDOM_STATE)
logreg_ridge = LogisticRegression(penalty="l2", solver="saga", random_state=RANDOM_STATE)
logreg_elasticnet = LogisticRegression(penalty="elasticnet", solver="saga", l1_ratio=0.5, random_state=RANDOM_STATE)
dt = DecisionTreeClassifier(random_state=RANDOM_STATE)
knn = KNeighborsClassifier()
rf = RandomForestClassifier(random_state=RANDOM_STATE)
ab = AdaBoostClassifier(random_state=RANDOM_STATE)
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
xgb = XGBClassifier(random_state=RANDOM_STATE)


In [13]:
models = {
    "LogisticRegressionBase": logreg_base,
    "LogisticRegressionLasso": logreg_lasso,
    "LogisticRegressionRidge": logreg_ridge,
    "LogisticRegressionElasticNet": logreg_elasticnet,
    "DecisionTree": dt,
    "KNearestNeigbor": knn,
    "RandomForest": rf,
    "AdaBoost": ab,
    "GradientBoost": gb,
    "XGBoost": xgb
}

results = benchmark_models(pipeline, models, x_train, y_train, x_test, y_test, 5, "Base")

In [16]:
results = results.sort_values("mean_ap_validate_score", ascending=False)
results[0:10]

,name,ap_test_score,mean_ap_train_score,mean_ap_validate_score,f1_test_score,recall_test_score,precision_test_score,accuracy_test_score
0,LogisticRegressionBase_Base,0.104564,0.087203,0.083697,0.000000,0.000000,0.000000,0.982820
1,LogisticRegressionElasticNet_Base,0.105818,0.087605,0.083485,0.000000,0.000000,0.000000,0.982820
2,LogisticRegressionLasso_Base,0.106671,0.086969,0.083256,0.000000,0.000000,0.000000,0.982820
3,LogisticRegressionRidge_Base,0.105050,0.087799,0.082672,0.000000,0.000000,0.000000,0.982820
4,GradientBoost_Base,0.098519,0.217658,0.076417,0.000000,0.000000,0.000000,0.982311
5,AdaBoost_Base,0.074363,0.071777,0.069321,0.000000,0.000000,0.000000,0.982820
6,XGBoost_Base,0.071300,0.595988,0.061618,0.000000,0.000000,0.000000,0.982566
7,RandomForest_Base,0.045699,0.952236,0.045896,0.034884,0.022222,0.081081,0.978875
8,KNearestNeigbor_Base,0.025167,0.247753,0.026020,0.014085,0.007407,0.142857,0.982184
9,DecisionTree_Base,0.027447,0.961599,0.023758,0.067568,0.074074,0.062112,0.964877


In [18]:
results.to_csv(SAVE_RESULT, index=False)